# Step 10 — Save and reproduce the pipeline

This notebook saves the information needed to rerun and audit the startup
diagnostic project:

1. cleaning rules;
2. analysis assumptions;
3. data dictionaries;
4. analysis-code inventory;
5. automated quality checks;
6. environment versions;
7. data and final-output manifests.

It also verifies the simple rerun command:

```bash
python src/run_pipeline.py
```

## Standard project structure

```text
project/
├── data/
│   ├── raw/
│   ├── cleaned/
│   └── processed/
├── notebooks/
├── src/
├── documentation/
└── outputs/
    ├── figures/
    ├── quality/
    └── tables/
```

Raw files are never overwritten. Cleaned, processed, and output files can
be regenerated from the saved code.

In [1]:
# Import standard libraries and the analysis libraries used by the project.
from pathlib import Path
import hashlib
import platform
import sys
import pandas as pd
import numpy as np
import matplotlib
import seaborn as sns
import sklearn

PIPELINE_VERSION = "1.0"
RANDOM_SEED = 42

# Find the project root when run from the root or notebooks folder.
BASE_DIR = Path.cwd()
if BASE_DIR.name == "notebooks":
    BASE_DIR = BASE_DIR.parent
elif not (BASE_DIR / "data").exists():
    if (BASE_DIR / "step4_test_project" / "data").exists():
        BASE_DIR = BASE_DIR / "step4_test_project"

RAW_DIR = BASE_DIR / "data" / "raw"
CLEANED_DIR = BASE_DIR / "data" / "cleaned"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
NOTEBOOK_DIR = BASE_DIR / "notebooks"
SRC_DIR = BASE_DIR / "src"
DOCS_DIR = BASE_DIR / "documentation"
OUTPUTS_DIR = BASE_DIR / "outputs"
TABLE_DIR = OUTPUTS_DIR / "tables"
QUALITY_DIR = OUTPUTS_DIR / "quality"

for folder in [DOCS_DIR, QUALITY_DIR]:
    folder.mkdir(parents=True, exist_ok=True)


def file_sha256(file_path):
    # Hashes make it possible to detect a changed input, script, or output.
    digest = hashlib.sha256()
    with open(file_path, "rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


print("Project root:", BASE_DIR)
print("Pipeline version:", PIPELINE_VERSION)

Project root: /workspace/scratch/5e59357ddcee/step4_test_project
Pipeline version: 1.0


## Load the final analysis tables

In [2]:
processed_files = {
    "crunchbase_feature_table":
        PROCESSED_DIR / "crunchbase_feature_table.csv",
    "failure_feature_table":
        PROCESSED_DIR / "failure_feature_table.csv",
    "startup_metrics_feature_table":
        PROCESSED_DIR / "startup_metrics_feature_table.csv",
}

required_output_files = {
    "validation_summary": TABLE_DIR / "step8_validation_summary.csv",
    "diagnostic_summary": TABLE_DIR / "step9_diagnostic_summary.csv",
    "startup_diagnostics":
        TABLE_DIR / "step9_startup_health_diagnostics.csv",
    "sector_diagnostics":
        TABLE_DIR / "step9_sector_failure_diagnostics.csv",
    "funding_diagnostics": TABLE_DIR / "step9_funding_diagnostics.csv",
}

required_files = list(processed_files.values()) + list(
    required_output_files.values()
)
missing_files = [str(file) for file in required_files if not file.exists()]

if missing_files:
    raise FileNotFoundError(
        f"Missing pipeline files: {missing_files}. Run earlier steps first."
    )

crunchbase = pd.read_csv(
    processed_files["crunchbase_feature_table"], low_memory=False
)
failure = pd.read_csv(
    processed_files["failure_feature_table"], low_memory=False
)
startup_metrics = pd.read_csv(
    processed_files["startup_metrics_feature_table"]
)

print(f"Crunchbase feature rows: {len(crunchbase):,}")
print(f"Failure feature rows:    {len(failure):,}")
print(f"Startup metric rows:     {len(startup_metrics):,}")

Crunchbase feature rows: 66,368
Failure feature rows:    409
Startup metric rows:     200


**Note.** The documentation below describes the final pipeline rules, not
only the current results. Update the version number whenever a rule or
assumption changes.

# 1. Cleaning rules

In [3]:
cleaning_rules = pd.DataFrame([
    {
        "rule_id": "C01",
        "dataset": "All datasets",
        "field": "All rows",
        "rule": "Remove exact duplicate rows.",
        "missing_value_policy": "Not applicable",
        "reason": "Prevents double counting identical records.",
    },
    {
        "rule_id": "C02",
        "dataset": "Crunchbase",
        "field": "permalink",
        "rule": "Keep the first row for a duplicated permalink.",
        "missing_value_policy": "Identifier should be present",
        "reason": "Permalink is the company-level identifier.",
    },
    {
        "rule_id": "C03",
        "dataset": "Synthetic metrics",
        "field": "startup_id",
        "rule": "Trim, uppercase, and keep the first duplicated ID.",
        "missing_value_policy": "Identifier should be present",
        "reason": "One row is required per synthetic startup.",
    },
    {
        "rule_id": "C04",
        "dataset": "All company tables",
        "field": "Company name",
        "rule": "Trim text and create a lowercase alphanumeric name key.",
        "missing_value_policy": "Keep missing names flagged",
        "reason": "Supports review of possible matches without automatic joins.",
    },
    {
        "rule_id": "C05",
        "dataset": "Crunchbase",
        "field": "status",
        "rule": "Trim and convert status to lowercase.",
        "missing_value_policy": "Keep missing status for review",
        "reason": "Creates consistent operating, closed, acquired, and IPO labels.",
    },
    {
        "rule_id": "C06",
        "dataset": "Crunchbase",
        "field": "category_list",
        "rule": "Trim separators and use the first category as primary_category.",
        "missing_value_policy": "Use explicit Unknown category",
        "reason": "Supports consistent category and peer-cohort analysis.",
    },
    {
        "rule_id": "C07",
        "dataset": "Crunchbase",
        "field": "Location fields",
        "rule": "Trim text; uppercase country code.",
        "missing_value_policy": "Use explicit Unknown text",
        "reason": "Preserves records while making missing geography visible.",
    },
    {
        "rule_id": "C08",
        "dataset": "Crunchbase",
        "field": "funding_total_usd",
        "rule": "Convert to numeric US dollars.",
        "missing_value_policy": "Dash or invalid text becomes missing, never zero",
        "reason": "Zero funding and unreported funding have different meanings.",
    },
    {
        "rule_id": "C09",
        "dataset": "Crunchbase",
        "field": "Date fields",
        "rule": "Parse founding and funding dates with invalid values as missing.",
        "missing_value_policy": "Keep missing date",
        "reason": "Avoids inventing dates and enables date-order checks.",
    },
    {
        "rule_id": "C10",
        "dataset": "Failure tables",
        "field": "Failure flags",
        "rule": "Convert measured flags to binary 0 or 1.",
        "missing_value_policy":
            "A flag absent from a source file remains missing, not zero",
        "reason": "Missing coverage is not evidence that a factor was absent.",
    },
    {
        "rule_id": "C11",
        "dataset": "Failure tables",
        "field": "How Much They Raised",
        "rule": "Parse K, M, and B suffixes into numeric US dollars.",
        "missing_value_policy": "Unclear values remain missing and are flagged",
        "reason": "Preserves the original text while enabling numeric analysis.",
    },
    {
        "rule_id": "C12",
        "dataset": "Failure tables",
        "field": "Years of Operation",
        "rule": "Extract the first and last four-digit years.",
        "missing_value_policy": "Unparseable years remain missing",
        "reason": "Supports approximate operating-duration analysis.",
    },
    {
        "rule_id": "C13",
        "dataset": "Synthetic metrics",
        "field": "Numeric metrics",
        "rule": "Convert metrics to numeric and replace negative values with missing.",
        "missing_value_policy": "Invalid or negative values remain missing",
        "reason": "Negative values are invalid for the supplied operating metrics.",
    },
    {
        "rule_id": "C14",
        "dataset": "Synthetic metrics",
        "field": "LTV/CAC and runway",
        "rule": "Recalculate and compare with supplied values.",
        "missing_value_policy": "Invalid calculations make the row ineligible",
        "reason": "Confirms the two core diagnostic measures.",
    },
    {
        "rule_id": "C15",
        "dataset": "All datasets",
        "field": "Cross-dataset matches",
        "rule": "Do not join datasets unless company identity is manually verified.",
        "missing_value_policy": "Not applicable",
        "reason": "A normalized name alone is not a reliable company match.",
    },
])

cleaning_rules["pipeline_version"] = PIPELINE_VERSION
cleaning_rules.to_csv(DOCS_DIR / "cleaning_rules.csv", index=False)

print(cleaning_rules.to_string(index=False))
print("\nSaved: documentation/cleaning_rules.csv")

rule_id            dataset                 field                                                                 rule                                       missing_value_policy                                                          reason pipeline_version
    C01       All datasets              All rows                                         Remove exact duplicate rows.                                             Not applicable                     Prevents double counting identical records.              1.0
    C02         Crunchbase             permalink                       Keep the first row for a duplicated permalink.                               Identifier should be present                      Permalink is the company-level identifier.              1.0
    C03  Synthetic metrics            startup_id                   Trim, uppercase, and keep the first duplicated ID.                               Identifier should be present                      One row is required per synt

# 2. Analysis assumptions

In [4]:
analysis_assumptions = pd.DataFrame([
    ["A01", "Company age", "Reference date is 2015-12-31, the end of the latest first-funding year.", "Company age is not confirmed operating lifespan."],
    ["A02", "Funding missingness", "Missing funding is unknown, not zero.", "Companies without reported funding are excluded from funding benchmarks."],
    ["A03", "Failure flag coverage", "An unmeasured failure flag remains missing.", "Sector prevalence uses available measurements and requires caution."],
    ["A04", "Failure population", "The detailed failure dataset contains failed startups only.", "It cannot estimate an active startup's probability of failure."],
    ["A05", "Observed status", "Operating, closed, acquired, and IPO are observed labels.", "Operating is not proof of success; acquisition is not always positive."],
    ["A06", "Observed exit", "Acquired or IPO is grouped as an observed exit.", "This is a descriptive outcome, not a complete success definition."],
    ["A07", "Health data", "The operating-metrics dataset is synthetic.", "Health counts demonstrate the method, not real population risk."],
    ["A08", "High health risk", "High Risk means runway under 6 months or LTV/CAC at or below 2.", "Thresholds are screening rules, not universal laws."],
    ["A09", "Healthy status", "Healthy means runway at least 12 months and LTV/CAC above 3.", "Other product and market risks can still exist."],
    ["A10", "Health ranking", "Runway and unit economics use three severity levels; runway wins an exact tie.", "The tie rule follows the more frequent validated signal."],
    ["A11", "Failure archetype", "Exact combinations preserve multi-factor failure patterns.", "Combinations are descriptive and can be fragmented."],
    ["A12", "Peer cohort", "Funding peers share primary category, country, and founding-period cohort.", "Different peer definitions can change benchmarks."],
    ["A13", "Eligible cohort", "Funding benchmark requires at least 30 funded records and 5 observed exits.", "This removes very small or outcome-sparse cohorts."],
    ["A14", "Funding position", "Bottom 25%, middle 50%, and top 25% are based on within-cohort funding percentile.", "Relative funding does not establish cash adequacy."],
    ["A15", "Class imbalance", "Funding holdout testing uses stratification and balanced class weights.", "Balanced metrics are reported instead of accuracy alone."],
    ["A16", "Uncertainty", "Bootstrap intervals use 2,000 resamples and random seed 42.", "Intervals describe sampling uncertainty, not all data bias."],
    ["A17", "Causality", "All findings use observational or synthetic data.", "Use associated with, co-occurs, or flags; avoid causal claims."],
    ["A18", "Dataset separation", "The three principal datasets remain separate.", "Integrated client diagnosis requires verified identity and current client data."],
], columns=["assumption_id", "topic", "assumption", "interpretation_limit"])

analysis_assumptions["pipeline_version"] = PIPELINE_VERSION
analysis_assumptions.to_csv(
    DOCS_DIR / "analysis_assumptions.csv", index=False
)

print(analysis_assumptions.to_string(index=False))
print("\nSaved: documentation/analysis_assumptions.csv")

assumption_id                 topic                                                                         assumption                                                            interpretation_limit pipeline_version
          A01           Company age            Reference date is 2015-12-31, the end of the latest first-funding year.                                Company age is not confirmed operating lifespan.              1.0
          A02   Funding missingness                                              Missing funding is unknown, not zero.        Companies without reported funding are excluded from funding benchmarks.              1.0
          A03 Failure flag coverage                                        An unmeasured failure flag remains missing.             Sector prevalence uses available measurements and requires caution.              1.0
          A04    Failure population                        The detailed failure dataset contains failed startups only.                  

# 3. Data dictionaries

In [5]:
# Descriptions for core source and engineered fields.
column_descriptions = {
    "permalink": "Unique Crunchbase company identifier.",
    "failure_record_id": "Pipeline-generated unique failure-record identifier.",
    "startup_id": "Unique synthetic startup identifier.",
    "name": "Company name from the source.",
    "Name": "Company name from the failure source.",
    "name_normalized": "Lowercase alphanumeric company name used only for match review.",
    "category_list": "Pipe-separated Crunchbase category list.",
    "primary_category": "First category in category_list.",
    "Sector": "Failure dataset industry sector.",
    "stage": "Synthetic startup funding or development stage.",
    "category_group": "Broad synthetic startup category.",
    "funding_total_usd": "Reported total company funding in US dollars.",
    "funding_raised_usd": "Parsed failure-record funding amount in US dollars.",
    "funding_total_missing": "True when total funding was not reported.",
    "funding_needs_review": "True when failure-record funding text is unclear.",
    "funding_rounds": "Reported number of funding rounds.",
    "status": "Observed company status: operating, closed, acquired, or IPO.",
    "country_code": "Standardized uppercase country code or UNKNOWN.",
    "state_code": "State or regional code from the source.",
    "region": "Source region label.",
    "city": "Source city label.",
    "founded_at": "Parsed company founding date.",
    "first_funding_at": "Parsed first-funding date.",
    "last_funding_at": "Parsed last-funding date.",
    "date_after_reference": "True when a relevant date exceeds the dataset reference date.",
    "funding_before_founding": "True when first funding precedes founding.",
    "company_age_years": "Years from founding to the 2015-12-31 reference date.",
    "funding_delay_months": "Months from founding to first funding.",
    "funding_span_months": "Months from first to last funding.",
    "funding_band": "Business-friendly funding range.",
    "founded_year": "Year extracted from founded_at.",
    "founded_cohort": "Five-year founding-period group.",
    "peer_cohort": "Dataset-specific peer-group key.",
    "peer_cohort_size": "Number of records in the peer cohort.",
    "Years of Operation": "Original operating-years text from failure source.",
    "What They Did": "Source description of the failed startup's activity.",
    "How Much They Raised": "Original funding text from the failure source.",
    "Why They Failed": "Source failure explanation.",
    "Takeaway": "Source post-mortem takeaway.",
    "start_year": "First year parsed from Years of Operation.",
    "end_year": "Last year parsed from Years of Operation.",
    "duplicate_name": "True when a normalized name repeats inside the source table.",
    "source_file": "Detailed failure source file.",
    "name_appears_multiple_times": "True when normalized name repeats in the combined failure table.",
    "failure_factor_count": "Number of recorded binary failure factors equal to 1.",
    "risk_combination": "Exact combination of recorded failure factors.",
    "operating_years": "Approximate end year minus start year.",
    "mrr_usd": "Monthly recurring revenue in US dollars.",
    "cac_usd": "Customer acquisition cost in US dollars.",
    "ltv_usd": "Customer lifetime value in US dollars.",
    "ltv_cac_ratio": "Supplied lifetime-value to acquisition-cost ratio.",
    "monthly_burn_usd": "Monthly cash burn in US dollars.",
    "cash_in_bank_usd": "Cash balance in US dollars.",
    "runway_months": "Supplied cash runway in months.",
    "health_status": "Rule-based synthetic health category.",
    "ltv_cac_ratio_calculated": "Recalculated LTV divided by CAC.",
    "ltv_cac_valid": "True when supplied and recalculated LTV/CAC agree within tolerance.",
    "runway_months_calculated": "Recalculated cash divided by monthly burn.",
    "runway_valid": "True when supplied and recalculated runway agree within tolerance.",
    "analysis_eligible": "True when required synthetic metrics pass validation.",
    "runway_band": "Practical cash-runway category.",
}

failure_flags = [
    "Giants", "No Budget", "Competition", "Poor Market Fit",
    "Acquisition Stagnation", "Platform Dependency",
    "High Operational Costs", "Monetization Failure", "Niche Limits",
    "Execution Flaws", "Trend Shifts", "Toxicity/Trust Issues",
    "Regulatory Pressure", "Overhype",
]


def column_role(column, dtype):
    if column in ["permalink", "failure_record_id", "startup_id"]:
        return "Identifier"
    if column in failure_flags:
        return "Binary failure factor"
    if (
        column.endswith("_valid")
        or column.endswith("_missing")
        or column.startswith("date_after")
        or column in [
            "analysis_eligible", "funding_before_founding",
            "funding_needs_review", "duplicate_name",
            "name_appears_multiple_times",
        ]
    ):
        return "Quality flag"
    if pd.api.types.is_numeric_dtype(dtype):
        return "Measure"
    return "Dimension or text"


def column_unit(column):
    if column.endswith("_usd"):
        return "USD"
    if "months" in column:
        return "Months"
    if "years" in column or column.endswith("_year"):
        return "Years"
    if column in failure_flags:
        return "0, 1, or missing"
    return "Not applicable"


dictionary_rows = []
dictionary_tables = {
    "crunchbase_feature_table.csv": crunchbase,
    "failure_feature_table.csv": failure,
    "startup_metrics_feature_table.csv": startup_metrics,
}

for table_name, table in dictionary_tables.items():
    for column in table.columns:
        description = column_descriptions.get(
            column,
            "Source field retained from cleaned data."
            if column not in failure_flags
            else f"Binary indicator for the recorded {column} factor.",
        )
        dictionary_rows.append({
            "table": table_name,
            "column": column,
            "data_type": str(table[column].dtype),
            "role": column_role(column, table[column].dtype),
            "unit": column_unit(column),
            "description": description,
            "missing_count": int(table[column].isna().sum()),
            "missing_pct": round(100 * table[column].isna().mean(), 2),
            "unique_values": int(table[column].nunique(dropna=True)),
        })

data_dictionary = pd.DataFrame(dictionary_rows)
data_dictionary["pipeline_version"] = PIPELINE_VERSION
data_dictionary.to_csv(DOCS_DIR / "data_dictionary.csv", index=False)

dictionary_summary = (
    data_dictionary.groupby("table")
    .agg(columns=("column", "size"), columns_with_missing=("missing_count", lambda values: int((values > 0).sum())))
    .reset_index()
)

print(dictionary_summary.to_string(index=False))
print(f"\nDictionary rows saved: {len(data_dictionary):,}")
print("Saved: documentation/data_dictionary.csv")

                            table  columns  columns_with_missing
     crunchbase_feature_table.csv       26                     9
        failure_feature_table.csv       36                     6
startup_metrics_feature_table.csv       19                     0

Dictionary rows saved: 81
Saved: documentation/data_dictionary.csv


# 4. Automated quality checks

In [6]:
check_rows = []


def add_check(check_id, area, check, passed, observed, expected):
    check_rows.append({
        "check_id": check_id,
        "area": area,
        "check": check,
        "status": "PASS" if bool(passed) else "FAIL",
        "observed": str(observed),
        "expected": str(expected),
    })


add_check("Q01", "Crunchbase", "Expected row count", len(crunchbase) == 66368, len(crunchbase), 66368)
add_check("Q02", "Crunchbase", "Unique permalink", not crunchbase["permalink"].duplicated().any(), int(crunchbase["permalink"].duplicated().sum()), 0)
add_check("Q03", "Crunchbase", "Valid observed statuses", set(crunchbase["status"].dropna().unique()).issubset({"operating", "closed", "acquired", "ipo"}), sorted(crunchbase["status"].dropna().unique()), "operating, closed, acquired, ipo")
add_check("Q04", "Crunchbase", "No negative reported funding", not (crunchbase["funding_total_usd"].dropna() < 0).any(), int((crunchbase["funding_total_usd"].dropna() < 0).sum()), 0)
add_check("Q05", "Crunchbase", "No negative valid funding delay", not (crunchbase["funding_delay_months"].dropna() < 0).any(), int((crunchbase["funding_delay_months"].dropna() < 0).sum()), 0)
add_check("Q06", "Failure", "Expected row count", len(failure) == 409, len(failure), 409)
add_check("Q07", "Failure", "Unique failure record ID", not failure["failure_record_id"].duplicated().any(), int(failure["failure_record_id"].duplicated().sum()), 0)

failure_flag_values = failure[failure_flags].stack().dropna()
invalid_failure_flags = int((~failure_flag_values.isin([0, 1])).sum())
add_check("Q08", "Failure", "Failure flags are binary or missing", invalid_failure_flags == 0, invalid_failure_flags, 0)

calculated_factor_count = failure[failure_flags].fillna(0).sum(axis=1)
factor_count_mismatches = int((calculated_factor_count != failure["failure_factor_count"]).sum())
add_check("Q09", "Failure", "Failure-factor count reproduces", factor_count_mismatches == 0, factor_count_mismatches, 0)
add_check("Q10", "Metrics", "Expected row count", len(startup_metrics) == 200, len(startup_metrics), 200)
add_check("Q11", "Metrics", "Unique startup ID", not startup_metrics["startup_id"].duplicated().any(), int(startup_metrics["startup_id"].duplicated().sum()), 0)
add_check("Q12", "Metrics", "All LTV/CAC calculations valid", startup_metrics["ltv_cac_valid"].fillna(False).all(), int((~startup_metrics["ltv_cac_valid"].fillna(False)).sum()), 0)
add_check("Q13", "Metrics", "All runway calculations valid", startup_metrics["runway_valid"].fillna(False).all(), int((~startup_metrics["runway_valid"].fillna(False)).sum()), 0)
add_check("Q14", "Metrics", "All rows analysis eligible", startup_metrics["analysis_eligible"].fillna(False).all(), int((~startup_metrics["analysis_eligible"].fillna(False)).sum()), 0)

startup_diagnostics = pd.read_csv(required_output_files["startup_diagnostics"])
sector_diagnostics = pd.read_csv(required_output_files["sector_diagnostics"])
funding_diagnostics = pd.read_csv(required_output_files["funding_diagnostics"], low_memory=False)

required_diagnostic_columns = [
    "bottleneck_rank", "ranked_bottleneck", "supporting_evidence",
    "possible_causes", "recommended_test", "KPI",
]
missing_diagnostic_values = sum(
    int(table[required_diagnostic_columns].isna().sum().sum())
    for table in [startup_diagnostics, sector_diagnostics, funding_diagnostics]
)
add_check("Q15", "Final outputs", "Required diagnostic fields are complete", missing_diagnostic_values == 0, missing_diagnostic_values, 0)
add_check("Q16", "Final outputs", "Startup diagnostics cover 200 startups", startup_diagnostics["startup_id"].nunique() == 200, startup_diagnostics["startup_id"].nunique(), 200)
add_check("Q17", "Final outputs", "Three ranked risks per sector", (sector_diagnostics.groupby("Sector").size() == 3).all(), sector_diagnostics.groupby("Sector").size().to_dict(), "3 per sector")
add_check("Q18", "Final outputs", "Funding diagnostics reproduce eligible population", len(funding_diagnostics) == 15878, len(funding_diagnostics), 15878)
add_check("Q19", "Pipeline", "Rerun script exists", (SRC_DIR / "run_pipeline.py").exists(), (SRC_DIR / "run_pipeline.py").exists(), True)

quality_checks = pd.DataFrame(check_rows)
quality_checks["pipeline_version"] = PIPELINE_VERSION
quality_checks.to_csv(
    QUALITY_DIR / "quality_check_results.csv", index=False
)

print(quality_checks.to_string(index=False))
print()
print(quality_checks["status"].value_counts().to_string())

if not quality_checks["status"].eq("PASS").all():
    failed = quality_checks[quality_checks["status"] == "FAIL"]
    raise ValueError(f"Quality checks failed:\n{failed.to_string(index=False)}")

check_id          area                                             check status                                                                                                                                      observed                         expected pipeline_version
     Q01    Crunchbase                                Expected row count   PASS                                                                                                                                         66368                            66368              1.0
     Q02    Crunchbase                                  Unique permalink   PASS                                                                                                                                             0                                0              1.0
     Q03    Crunchbase                           Valid observed statuses   PASS                                                                                                    ['acq

**Quality result.** The pipeline stops when a required check fails. Missing
values that are expected by design, such as unreported funding or
unmeasured failure flags, are documented rather than treated as errors.

# 5. Analysis-code inventory

In [7]:
expected_code_files = [
    ("Step 2", SRC_DIR / "01_data_quality_audit.py", "Audit raw CSV files"),
    ("Step 3", SRC_DIR / "02_clean_and_standardize.py", "Clean and standardize raw files"),
    ("Runner", SRC_DIR / "run_pipeline.py", "Run the complete pipeline"),
    ("Step 4", NOTEBOOK_DIR / "Step_4_Build_Analysis_Ready_Tables.ipynb", "Build separate analysis tables"),
    ("Step 5", NOTEBOOK_DIR / "Step_5_Engineer_Useful_Features.ipynb", "Engineer analysis features"),
    ("Step 6", NOTEBOOK_DIR / "04_Exploratory_Analysis_Reviewed.ipynb", "Perform exploratory analysis"),
    ("Step 7", NOTEBOOK_DIR / "05_Answer_Three_Diagnostic_Questions.ipynb", "Answer the diagnostic questions"),
    ("Step 8", NOTEBOOK_DIR / "06_Validate_Findings.ipynb", "Validate findings"),
    ("Step 9", NOTEBOOK_DIR / "07_Convert_Findings_Into_Diagnostics.ipynb", "Create diagnostic outputs"),
    ("Step 10", NOTEBOOK_DIR / "08_Save_And_Reproduce_Pipeline.ipynb", "Document and verify reproducibility"),
]

code_inventory_rows = []
for step, file_path, purpose in expected_code_files:
    exists = file_path.exists()
    code_inventory_rows.append({
        "step": step,
        "file": str(file_path.relative_to(BASE_DIR)),
        "purpose": purpose,
        "exists": exists,
        "size_bytes": file_path.stat().st_size if exists else pd.NA,
        "sha256": file_sha256(file_path) if exists else pd.NA,
    })

analysis_code_inventory = pd.DataFrame(code_inventory_rows)
analysis_code_inventory["pipeline_version"] = PIPELINE_VERSION
analysis_code_inventory.to_csv(
    DOCS_DIR / "analysis_code_inventory.csv", index=False
)

print(analysis_code_inventory[[
    "step", "file", "purpose", "exists"
]].to_string(index=False))

if not analysis_code_inventory["exists"].all():
    raise FileNotFoundError("One or more required pipeline code files are missing.")

   step                                                 file                             purpose  exists
 Step 2                         src/01_data_quality_audit.py                 Audit raw CSV files    True
 Step 3                      src/02_clean_and_standardize.py     Clean and standardize raw files    True
 Runner                                  src/run_pipeline.py           Run the complete pipeline    True
 Step 4   notebooks/Step_4_Build_Analysis_Ready_Tables.ipynb      Build separate analysis tables    True
 Step 5      notebooks/Step_5_Engineer_Useful_Features.ipynb          Engineer analysis features    True
 Step 6     notebooks/04_Exploratory_Analysis_Reviewed.ipynb        Perform exploratory analysis    True
 Step 7 notebooks/05_Answer_Three_Diagnostic_Questions.ipynb     Answer the diagnostic questions    True
 Step 8                 notebooks/06_Validate_Findings.ipynb                   Validate findings    True
 Step 9 notebooks/07_Convert_Findings_Into_Diagnostics.

# 6. Data, environment, and final-output manifests

In [8]:
# Record every raw, cleaned, and processed CSV without copying the data.
data_manifest_rows = []
for stage, folder in [
    ("raw", RAW_DIR),
    ("cleaned", CLEANED_DIR),
    ("processed", PROCESSED_DIR),
]:
    for file_path in sorted(folder.glob("*.csv")):
        data_manifest_rows.append({
            "stage": stage,
            "file": str(file_path.relative_to(BASE_DIR)),
            "size_bytes": file_path.stat().st_size,
            "sha256": file_sha256(file_path),
        })

data_file_manifest = pd.DataFrame(data_manifest_rows)
data_file_manifest["pipeline_version"] = PIPELINE_VERSION
data_file_manifest.to_csv(
    DOCS_DIR / "data_file_manifest.csv", index=False
)

# Record all outputs that existed before this manifest was written.
output_manifest_file = DOCS_DIR / "final_outputs_manifest.csv"
output_rows = []
for file_path in sorted(OUTPUTS_DIR.rglob("*")):
    if not file_path.is_file() or file_path == output_manifest_file:
        continue
    output_rows.append({
        "file": str(file_path.relative_to(BASE_DIR)),
        "output_type": file_path.parent.name,
        "size_bytes": file_path.stat().st_size,
        "sha256": file_sha256(file_path),
    })

final_outputs_manifest = pd.DataFrame(output_rows)
final_outputs_manifest["pipeline_version"] = PIPELINE_VERSION
final_outputs_manifest.to_csv(output_manifest_file, index=False)

environment_versions = pd.DataFrame([
    ["Python", platform.python_version()],
    ["pandas", pd.__version__],
    ["numpy", np.__version__],
    ["matplotlib", matplotlib.__version__],
    ["seaborn", sns.__version__],
    ["scikit-learn", sklearn.__version__],
], columns=["package", "version"])
environment_versions.to_csv(
    DOCS_DIR / "environment_versions.csv", index=False
)

requirements_lines = [
    f"pandas=={pd.__version__}",
    f"numpy=={np.__version__}",
    f"matplotlib=={matplotlib.__version__}",
    f"seaborn=={sns.__version__}",
    f"scikit-learn=={sklearn.__version__}",
]
(BASE_DIR / "requirements.txt").write_text(
    "\n".join(requirements_lines) + "\n",
    encoding="utf-8",
)

print(f"Data files recorded:   {len(data_file_manifest):,}")
print(f"Output files recorded: {len(final_outputs_manifest):,}")
print()
print(environment_versions.to_string(index=False))

Data files recorded:   28
Output files recorded: 49

     package version
      Python 3.12.14
      pandas   2.2.3
       numpy   2.3.5
  matplotlib  3.10.8
     seaborn  0.13.2
scikit-learn   1.8.0


In [9]:
readme_lines = [
    "# Startup Diagnostic Pipeline",
    "",
    "## Run the complete pipeline",
    "",
    "From the project root, run:",
    "",
    "```bash",
    "python src/run_pipeline.py",
    "```",
    "",
    "To restart from a later step, for example Step 6:",
    "",
    "```bash",
    "python src/run_pipeline.py --from-step 6",
    "```",
    "",
    "## Execution order",
    "",
    "1. Audit raw data.",
    "2. Clean and standardize data.",
    "3. Build separate analysis tables.",
    "4. Engineer features.",
    "5. Perform exploratory analysis.",
    "6. Answer the diagnostic questions.",
    "7. Validate the findings.",
    "8. Convert findings into diagnostics.",
    "9. Refresh documentation and reproducibility checks.",
    "",
    "## Important limits",
    "",
    "- Do not merge the three principal datasets without verified company matches.",
    "- Failure data describes failed startups only.",
    "- Synthetic metrics demonstrate the method and do not estimate real-world risk.",
    "- Funding results are observational associations, not causal estimates.",
    "",
    "## Documentation",
    "",
    "See the documentation folder for cleaning rules, assumptions, data dictionaries, code hashes, data hashes, output hashes, and environment versions.",
]
(DOCS_DIR / "README_pipeline.md").write_text(
    "\n".join(readme_lines) + "\n",
    encoding="utf-8",
)

pipeline_summary = pd.DataFrame([
    ["Cleaning rules", DOCS_DIR / "cleaning_rules.csv", len(cleaning_rules)],
    ["Analysis assumptions", DOCS_DIR / "analysis_assumptions.csv", len(analysis_assumptions)],
    ["Data dictionary", DOCS_DIR / "data_dictionary.csv", len(data_dictionary)],
    ["Quality checks", QUALITY_DIR / "quality_check_results.csv", len(quality_checks)],
    ["Analysis code inventory", DOCS_DIR / "analysis_code_inventory.csv", len(analysis_code_inventory)],
    ["Data file manifest", DOCS_DIR / "data_file_manifest.csv", len(data_file_manifest)],
    ["Final output manifest", DOCS_DIR / "final_outputs_manifest.csv", len(final_outputs_manifest)],
    ["Environment versions", DOCS_DIR / "environment_versions.csv", len(environment_versions)],
], columns=["artifact", "file", "records"])
pipeline_summary["file"] = pipeline_summary["file"].map(
    lambda path: str(Path(path).relative_to(BASE_DIR))
)
pipeline_summary.to_csv(
    DOCS_DIR / "reproducibility_summary.csv", index=False
)

print(pipeline_summary.to_string(index=False))
print()
print("Pipeline documentation and manifests saved successfully.")

               artifact                                      file  records
         Cleaning rules          documentation/cleaning_rules.csv       15
   Analysis assumptions    documentation/analysis_assumptions.csv       18
        Data dictionary         documentation/data_dictionary.csv       81
         Quality checks outputs/quality/quality_check_results.csv       19
Analysis code inventory documentation/analysis_code_inventory.csv       10
     Data file manifest      documentation/data_file_manifest.csv       28
  Final output manifest  documentation/final_outputs_manifest.csv       49
   Environment versions    documentation/environment_versions.csv        6

Pipeline documentation and manifests saved successfully.


## Step 10 result

The pipeline is now documented and reproducible. The key command is:

```bash
python src/run_pipeline.py
```

Before interpreting refreshed results, confirm that every row in
`outputs/quality/quality_check_results.csv` has status `PASS`.

When a cleaning rule, threshold, cohort definition, or diagnostic rule is
changed, update the pipeline version and rerun the full pipeline so the
documentation, hashes, checks, and outputs remain aligned.